# Single-family transfer and entropy analysis

CPU only. Run top to bottom. Requires raw collections and the existing main protocol. Supplementary experiments use their own saved pools and results; do not mix their AUCs with the main experiment. Computationally heavier than table generation.

In [ ]:
from pathlib import Path
import os
import json
from IPython.display import display

# Edit this ONE setting if your collections live elsewhere.
DATA_ROOT = Path(os.environ.get("B3IT_DATA_ROOT", "/content/drive/MyDrive"))
if not DATA_ROOT.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        raise FileNotFoundError("Set B3IT_DATA_ROOT to the directory containing both model folders.")
assert DATA_ROOT.exists(), DATA_ROOT
collections = {
    "Qwen": DATA_ROOT / "B3IT_Qwen05_Noise_Pruning" / "expanded_scales_1000samples_v1",
    "Phi": DATA_ROOT / "B3IT_Phi4mini_Noise_Pruning" / "expanded_scales_1000samples_v1",
}
folders = {model: folder / "extension_selection_repeated_v1" for model, folder in collections.items()}


## Freeze/check supplementary protocol

In [ ]:
import json
from itertools import permutations
from pathlib import Path

for model, folder in collections.items():
    manifest = json.loads((folder / "manifest.json").read_text())
    cfg = manifest["config"]

    families = {
        kind: sorted(
            job["name"]
            for job in manifest["jobs"]
            if job["kind"] == kind
        )
        for kind in ["magnitude", "random", "random_noise"]
    }

    assert [len(families[k]) for k in families] == [5, 5, 7]

    protocol = {
        "experiment": "single_family_selection_v1",
        "model": cfg["model"],
        "seeds": [101, 202, 303, 404, 505],
        "candidate_prompts": cfg["prompts"],
        "families": families,
        "train_test_pairs": [
            {"train": train, "test": test}
            for train, test in permutations(families, 2)
        ],
        "selection_pool_size": 500,
        "evaluation_pool_size": 500,
        "split_method": (
            "Multivariate hypergeometric partition without replacement. "
            "Save selection and evaluation counts explicitly."
        ),
        "rng_method": (
            "Separate deterministic generators derived from SHA-256 "
            "of model, seed, operation and identifiers; "
            "independent of execution order."
        ),
        "selected_prompts": 3,
        "detection_samples_per_prompt": 10,
        "reference_samples_per_prompt": 50,
        "simulation_tests_per_class": 1000,
        "detection_requests_per_test": 30,
        "resampling": (
            "Independent multinomial reference and detection draws "
            "with replacement within the appropriate pool."
        ),
        "score": "Mean TV distance across selected inputs",
        "selection": (
            "Rank individual inputs by mean selection AUC over "
            "the chosen training variants; retain the top three."
        ),
        "tie_break": "Original candidate-prompt order",
        "evaluation": (
            "Evaluate all variants of the test family; "
            "average variant AUCs equally."
        ),
        "strategies": [
            "performance_based",
            "entropy_based",
            "uniform_random_all_120_subsets",
        ],
        "variant_budget_curve": {
            "sizes": "1 through the number of training variants",
            "subsets": (
                "All combinations at each size; "
                "no random subsampling of variant combinations."
            ),
            "summary": (
                "Average equally across training-variant combinations "
                "within each seed, then summarize across seeds."
            ),
        },
        "entropy_analysis": (
            "Spearman correlation between unchanged selection-pool "
            "entropy and individual-input selection AUC, "
            "using all variants of each training family. "
            "Descriptive analysis on ten inputs; no causal claim."
        ),
        "comparisons": (
            "Recompute all strategies on the new evaluation pools. "
            "Do not combine with previous experiment AUCs."
        ),
        "cost_scope": (
            "Detection observations only; discovery, selection "
            "and reference collection excluded."
        ),
        "limitations": [
            "Existing ten-input candidate pools",
            "Repeated partitions of the same recorded outputs",
            "Descriptive seed SD, not confidence intervals",
            "Supplementary experiment designed after earlier results",
        ],
    }

    output = folder / "extension_single_family_v1"
    output.mkdir(exist_ok=True)
    path = output / "protocol.json"

    if path.exists():
        assert json.loads(path.read_text()) == protocol, (
            f"Different protocol already exists: {path}"
        )
        print(model, ": protocole identique déjà sauvegardé.")
    else:
        with path.open("x") as file:
            json.dump(protocol, file, indent=2, ensure_ascii=False)
        print(model, ": protocole sauvegardé.")

    print("  6 combinaisons entraînement/test")
    print("  Budget : 3 × 10 = 30 observations de détection")
    print(" ", path)

## Create/check disjoint saved pools

In [ ]:
import hashlib
import json
import numpy as np

def deterministic_rng(model_name, seed, operation, *identifiers):
    key = json.dumps(
        [model_name, seed, operation, *identifiers],
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")

    digest = hashlib.sha256(key).digest()
    return np.random.default_rng(int.from_bytes(digest, "big"))


def validate_pools(saved, originals, prompts):
    for name, original in originals.items():
        for prompt in prompts:
            source = original["counts"][prompt]
            selection = saved["selection"][name][prompt]
            evaluation = saved["evaluation"][name][prompt]

            assert set(selection) == set(evaluation) == set(source)
            assert sum(selection.values()) == 500
            assert sum(evaluation.values()) == 500

            for token, count in source.items():
                assert selection[token] >= 0
                assert evaluation[token] >= 0
                assert selection[token] + evaluation[token] == count


for model, folder in collections.items():
    output = folder / "extension_single_family_v1"
    protocol = json.loads((output / "protocol.json").read_text())
    manifest = json.loads((folder / "manifest.json").read_text())

    names = sorted(job["name"] for job in manifest["jobs"])
    prompts = protocol["candidate_prompts"]

    originals = {
        name: json.loads((folder / f"{name}.json").read_text())
        for name in names
    }

    # Fingerprint the counts used, independently of JSON formatting.
    source_counts = {
        name: originals[name]["counts"] for name in names
    }
    fingerprint = hashlib.sha256(
        json.dumps(
            source_counts,
            sort_keys=True,
            ensure_ascii=False,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()

    for name, original in originals.items():
        assert original["config"]["model"] == protocol["model"]
        assert set(original["counts"]) == set(prompts)

        for prompt in prompts:
            counts = original["counts"][prompt]
            assert all(
                isinstance(value, int) and value >= 0
                for value in counts.values()
            )
            assert sum(counts.values()) == 1000

    for seed in protocol["seeds"]:
        path = output / f"pools_seed_{seed}.json"

        if path.exists():
            saved = json.loads(path.read_text())
            assert saved["protocol"] == protocol
            assert saved["source_counts_sha256"] == fingerprint
            assert saved["seed"] == seed
            validate_pools(saved, originals, prompts)
            print(f"{model} — seed {seed} : déjà sauvegardée, vérifiée.")
            continue

        saved = {
            "seed": seed,
            "protocol": protocol,
            "source_counts_sha256": fingerprint,
            "selection": {},
            "evaluation": {},
        }

        for name in names:
            saved["selection"][name] = {}
            saved["evaluation"][name] = {}

            for prompt in prompts:
                counts = originals[name]["counts"][prompt]
                tokens = sorted(counts)
                total = np.array(
                    [counts[token] for token in tokens],
                    dtype=np.int64,
                )

                rng = deterministic_rng(
                    protocol["model"], seed, "output_split",
                    name, prompt,
                )
                selected = rng.multivariate_hypergeometric(
                    total, 500
                )
                evaluated = total - selected

                saved["selection"][name][prompt] = {
                    token: int(value)
                    for token, value in zip(tokens, selected)
                }
                saved["evaluation"][name][prompt] = {
                    token: int(value)
                    for token, value in zip(tokens, evaluated)
                }

        validate_pools(saved, originals, prompts)

        temporary = path.with_suffix(".tmp")
        temporary.write_text(
            json.dumps(saved, ensure_ascii=False)
        )
        temporary.replace(path)

        print(f"{model} — seed {seed} : pools créés et vérifiés.")

    print(f"Saved: {output}\n")

## Individual-input selection scores

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

def simulate_tv(reference_prob, detection_prob, rng, protocol):
    n_tests = protocol["simulation_tests_per_class"]
    n_reference = protocol["reference_samples_per_prompt"]
    n_detection = protocol["detection_samples_per_prompt"]

    reference = rng.multinomial(
        n_reference, reference_prob, size=n_tests
    ) / n_reference

    detection = rng.multinomial(
        n_detection, detection_prob, size=n_tests
    ) / n_detection

    return 0.5 * np.abs(reference - detection).sum(axis=1)


for model, folder in collections.items():
    output = folder / "extension_single_family_v1"
    protocol = json.loads((output / "protocol.json").read_text())
    prompts = protocol["candidate_prompts"]

    variants = [
        name
        for names in protocol["families"].values()
        for name in names
    ]

    for seed in protocol["seeds"]:
        pools = json.loads(
            (output / f"pools_seed_{seed}.json").read_text()
        )
        path = output / f"selection_scores_seed_{seed}.json"

        if path.exists():
            saved = json.loads(path.read_text())
            assert saved["protocol"] == protocol
            assert (
                saved["source_counts_sha256"]
                == pools["source_counts_sha256"]
            )
            assert saved["seed"] == seed
            print(f"{model} — seed {seed} : déjà calculée.")
            continue

        selection = pools["selection"]
        baseline = selection["unchanged"]

        scores = {name: {} for name in variants}
        entropies = {}

        for prompt in prompts:
            # Entropy of the unchanged selection distribution.
            counts = np.array(
                list(baseline[prompt].values()), dtype=float
            )
            probabilities = counts[counts > 0] / counts.sum()
            entropies[prompt] = float(
                -(probabilities * np.log2(probabilities)).sum()
            )

            # Shared token support for this input.
            tokens = sorted({
                token
                for counts_by_prompt in selection.values()
                for token in counts_by_prompt[prompt]
            })

            original_prob = np.array([
                baseline[prompt].get(token, 0)
                for token in tokens
            ], dtype=float)
            original_prob /= original_prob.sum()

            null_rng = deterministic_rng(
                protocol["model"], seed,
                "selection_null", prompt,
            )
            null_scores = simulate_tv(
                original_prob, original_prob,
                null_rng, protocol,
            )

            for name in variants:
                changed_prob = np.array([
                    selection[name][prompt].get(token, 0)
                    for token in tokens
                ], dtype=float)
                changed_prob /= changed_prob.sum()

                changed_rng = deterministic_rng(
                    protocol["model"], seed,
                    "selection_changed", name, prompt,
                )
                changed_scores = simulate_tv(
                    original_prob, changed_prob,
                    changed_rng, protocol,
                )

                labels = np.concatenate([
                    np.zeros(len(null_scores)),
                    np.ones(len(changed_scores)),
                ])
                values = np.concatenate([
                    null_scores, changed_scores,
                ])

                scores[name][prompt] = float(
                    roc_auc_score(labels, values)
                )

        family_scores = {}
        selected_prompts = {}

        for family, names in protocol["families"].items():
            family_scores[family] = {
                prompt: float(np.mean([
                    scores[name][prompt] for name in names
                ]))
                for prompt in prompts
            }

            # Stable sort: ties retain original prompt order.
            selected_prompts[family] = sorted(
                prompts,
                key=lambda prompt: -family_scores[family][prompt],
            )[:protocol["selected_prompts"]]

        entropy_prompts = sorted(
            prompts, key=lambda prompt: -entropies[prompt]
        )[:protocol["selected_prompts"]]

        saved = {
            "seed": seed,
            "protocol": protocol,
            "source_counts_sha256": pools["source_counts_sha256"],
            "auc_by_variant_and_prompt": scores,
            "entropy_bits": entropies,
            "family_ranking_scores": family_scores,
            "performance_prompts_by_training_family": selected_prompts,
            "entropy_prompts": entropy_prompts,
        }

        temporary = path.with_suffix(".tmp")
        temporary.write_text(
            json.dumps(saved, indent=2, ensure_ascii=False)
        )
        temporary.replace(path)

        print(f"{model} — seed {seed} : scores sauvegardés.")

    # Show selections for seed 101 as a quick check.
    saved = json.loads(
        (output / "selection_scores_seed_101.json").read_text()
    )

    print(f"\n{model} — sélection avec toutes les variantes :")
    display(pd.DataFrame([
        {
            "Training family": family,
            "Selected prompts": repr(selected),
        }
        for family, selected in saved[
            "performance_prompts_by_training_family"
        ].items()
    ]))

## Evaluate six transfer directions

In [ ]:
from itertools import combinations
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

all_results = []

for model, folder in collections.items():
    output = folder / "extension_single_family_v1"
    protocol = json.loads((output / "protocol.json").read_text())
    prompts = protocol["candidate_prompts"]
    subsets = list(combinations(
        range(len(prompts)), protocol["selected_prompts"]
    ))
    subset_lookup = {
        frozenset(prompts[i] for i in indices): index
        for index, indices in enumerate(subsets)
    }

    for seed in protocol["seeds"]:
        path = output / f"evaluation_seed_{seed}.json"
        pools = json.loads(
            (output / f"pools_seed_{seed}.json").read_text()
        )
        selection = json.loads(
            (output / f"selection_scores_seed_{seed}.json").read_text()
        )

        assert selection["protocol"] == protocol
        assert (
            selection["source_counts_sha256"]
            == pools["source_counts_sha256"]
        )

        if path.exists():
            saved = json.loads(path.read_text())
            assert saved["protocol"] == protocol
            assert saved["seed"] == seed
            assert (
                saved["source_counts_sha256"]
                == pools["source_counts_sha256"]
            )
            print(f"{model} — seed {seed} : déjà évaluée.", flush=True)

        else:
            evaluation = pools["evaluation"]
            baseline = evaluation["unchanged"]
            n_tests = protocol["simulation_tests_per_class"]
            labels = np.r_[np.zeros(n_tests), np.ones(n_tests)]

            null_matrix = np.empty((n_tests, len(prompts)))
            changed_matrices = {
                name: np.empty((n_tests, len(prompts)))
                for names in protocol["families"].values()
                for name in names
            }

            for column, prompt in enumerate(prompts):
                tokens = sorted({
                    token
                    for counts_by_prompt in evaluation.values()
                    for token in counts_by_prompt[prompt]
                })

                original_prob = np.array([
                    baseline[prompt].get(token, 0) for token in tokens
                ], dtype=float)
                original_prob /= original_prob.sum()

                null_matrix[:, column] = simulate_tv(
                    original_prob,
                    original_prob,
                    deterministic_rng(
                        protocol["model"], seed,
                        "evaluation_null", prompt,
                    ),
                    protocol,
                )

                for name, matrix in changed_matrices.items():
                    changed_prob = np.array([
                        evaluation[name][prompt].get(token, 0)
                        for token in tokens
                    ], dtype=float)
                    changed_prob /= changed_prob.sum()

                    matrix[:, column] = simulate_tv(
                        original_prob,
                        changed_prob,
                        deterministic_rng(
                            protocol["model"], seed,
                            "evaluation_changed", name, prompt,
                        ),
                        protocol,
                    )

            # Shared simulation scores ensure paired comparisons.
            null_group_scores = [
                null_matrix[:, indices].mean(axis=1)
                for indices in subsets
            ]

            auc_by_variant = {}
            for name, matrix in changed_matrices.items():
                auc_by_variant[name] = [
                    float(roc_auc_score(
                        labels,
                        np.r_[
                            null_group_scores[index],
                            matrix[:, indices].mean(axis=1),
                        ],
                    ))
                    for index, indices in enumerate(subsets)
                ]

            family_aucs = {
                family: np.mean([
                    auc_by_variant[name] for name in names
                ], axis=0)
                for family, names in protocol["families"].items()
            }

            entropy_index = subset_lookup[
                frozenset(selection["entropy_prompts"])
            ]

            summary = []
            for pair in protocol["train_test_pairs"]:
                train, test = pair["train"], pair["test"]
                chosen = selection[
                    "performance_prompts_by_training_family"
                ][train]
                chosen_index = subset_lookup[frozenset(chosen)]

                aucs = family_aucs[test]
                performance = float(aucs[chosen_index])
                entropy = float(aucs[entropy_index])
                random_mean = float(aucs.mean())

                summary.append({
                    "model": model,
                    "seed": seed,
                    "training_family": train,
                    "test_family": test,
                    "performance_auc": performance,
                    "entropy_auc": entropy,
                    "random_mean_auc": random_mean,
                    "gain_vs_entropy": performance - entropy,
                    "gain_vs_random": performance - random_mean,
                    "selected_prompts": chosen,
                })

            saved = {
                "seed": seed,
                "protocol": protocol,
                "source_counts_sha256": pools["source_counts_sha256"],
                "subsets": [
                    [prompts[i] for i in indices]
                    for indices in subsets
                ],
                "auc_by_variant": auc_by_variant,
                "summary": summary,
            }

            temporary = path.with_suffix(".tmp")
            temporary.write_text(
                json.dumps(saved, ensure_ascii=False)
            )
            temporary.replace(path)

            print(f"{model} — seed {seed} : évaluée et sauvegardée.",
                  flush=True)

        all_results.extend(saved["summary"])

results = pd.DataFrame(all_results)

for model, folder in collections.items():
    results[results["model"] == model].to_csv(
        folder / "extension_single_family_v1"
        / "single_family_all_seeds.csv",
        index=False,
    )

summary = (
    results.groupby(
        ["model", "training_family", "test_family"], sort=False
    )
    .agg(
        performance_mean=("performance_auc", "mean"),
        performance_sd=("performance_auc", "std"),
        entropy_mean=("entropy_auc", "mean"),
        random_mean=("random_mean_auc", "mean"),
        gain_vs_entropy=("gain_vs_entropy", "mean"),
        gain_vs_random=("gain_vs_random", "mean"),
        positive_vs_random=(
            "gain_vs_random", lambda x: int((x > 0).sum())
        ),
    )
    .reset_index()
)

for model, folder in collections.items():
    summary[summary["model"] == model].to_csv(
        folder / "extension_single_family_v1"
        / "single_family_summary.csv",
        index=False,
    )

display(summary.round(4))
print("SD : variation entre les cinq splits et simulations, pas un IC.")

## All training-variant combinations

In [ ]:
from itertools import combinations
import json
import numpy as np
import pandas as pd

curve_rows = []

for model, folder in collections.items():
    output = folder / "extension_single_family_v1"
    protocol = json.loads((output / "protocol.json").read_text())
    prompts = protocol["candidate_prompts"]

    for seed in protocol["seeds"]:
        selection = json.loads(
            (output / f"selection_scores_seed_{seed}.json").read_text()
        )
        evaluation = json.loads(
            (output / f"evaluation_seed_{seed}.json").read_text()
        )

        assert selection["protocol"] == evaluation["protocol"] == protocol
        assert (
            selection["source_counts_sha256"]
            == evaluation["source_counts_sha256"]
        )

        subset_lookup = {
            frozenset(subset): i
            for i, subset in enumerate(evaluation["subsets"])
        }

        # Test AUCs remain fixed for every training-variant budget.
        test_aucs = {
            family: np.mean([
                evaluation["auc_by_variant"][name]
                for name in names
            ], axis=0)
            for family, names in protocol["families"].items()
        }

        for train, variants in protocol["families"].items():
            for k in range(1, len(variants) + 1):
                combinations_by_test = {
                    test: []
                    for test in protocol["families"]
                    if test != train
                }

                for chosen_variants in combinations(variants, k):
                    ranking = {
                        prompt: np.mean([
                            selection["auc_by_variant_and_prompt"]
                            [name][prompt]
                            for name in chosen_variants
                        ])
                        for prompt in prompts
                    }

                    chosen_prompts = sorted(
                        prompts, key=lambda p: -ranking[p]
                    )[:protocol["selected_prompts"]]

                    index = subset_lookup[frozenset(chosen_prompts)]

                    for test in combinations_by_test:
                        combinations_by_test[test].append(
                            float(test_aucs[test][index])
                        )

                for test, values in combinations_by_test.items():
                    random_mean = float(test_aucs[test].mean())

                    curve_rows.append({
                        "model": model,
                        "seed": seed,
                        "training_family": train,
                        "test_family": test,
                        "n_training_variants": k,
                        "n_combinations": len(values),
                        "performance_auc": float(np.mean(values)),
                        "min_combination_auc": float(np.min(values)),
                        "max_combination_auc": float(np.max(values)),
                        "random_mean_auc": random_mean,
                        "gain_vs_random": (
                            float(np.mean(values)) - random_mean
                        ),
                    })

curve = pd.DataFrame(curve_rows)

curve_summary = (
    curve.groupby([
        "model", "training_family", "test_family",
        "n_training_variants",
    ], sort=False)
    .agg(
        mean_auc=("performance_auc", "mean"),
        sd_across_seeds=("performance_auc", "std"),
        mean_gain_vs_random=("gain_vs_random", "mean"),
        n_combinations=("n_combinations", "first"),
    )
    .reset_index()
)

for model, folder in collections.items():
    output = folder / "extension_single_family_v1"

    curve[curve["model"] == model].to_csv(
        output / "variant_budget_curve_per_seed.csv",
        index=False,
    )
    curve_summary[curve_summary["model"] == model].to_csv(
        output / "variant_budget_curve_summary.csv",
        index=False,
    )

# Check that using every training variant reproduces the last analysis.
previous = pd.concat([
    pd.read_csv(
        folder / "extension_single_family_v1"
        / "single_family_all_seeds.csv"
    )
    for folder in collections.values()
], ignore_index=True)

for model, folder in collections.items():
    protocol = json.loads(
        (folder / "extension_single_family_v1"
         / "protocol.json").read_text()
    )

    for train, variants in protocol["families"].items():
        final = curve[
            (curve["model"] == model)
            & (curve["training_family"] == train)
            & (curve["n_training_variants"] == len(variants))
        ]

        check = final.merge(
            previous,
            on=["model", "seed", "training_family", "test_family"],
            suffixes=("_curve", "_previous"),
            validate="one_to_one",
        )
        assert len(check) == 10
        assert np.allclose(
            check["performance_auc_curve"],
            check["performance_auc_previous"],
            rtol=0, atol=1e-12,
        )

print("Vérifié : les derniers points reproduisent les résultats précédents.")
print("\nSÉLECTION AVEC DU BRUIT GAUSSIEN")
display(
    curve_summary[
        curve_summary["training_family"] == "random_noise"
    ].round(4)
)
print("\nCourbes sauvegardées pour les six combinaisons et les deux modèles.")

## Gaussian-noise transfer figure

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fig, axes = plt.subplots(
    1, 2, figsize=(10, 4), sharey=True,
    constrained_layout=True,
)

labels = {
    "magnitude": "Magnitude pruning",
    "random": "Random pruning",
}

for ax, (model, folder) in zip(axes, collections.items()):
    output = folder / "extension_single_family_v1"
    data = pd.read_csv(
        output / "variant_budget_curve_per_seed.csv"
    )
    data = data[data["training_family"] == "random_noise"]

    for test, label in labels.items():
        stats = (
            data[data["test_family"] == test]
            .groupby("n_training_variants")["gain_vs_random"]
            .agg(["mean", "std"])
        )

        ax.errorbar(
            stats.index,
            stats["mean"],
            yerr=stats["std"],
            marker="o",
            capsize=3,
            label=label,
        )

    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.set_title("Qwen2.5-0.5B" if model == "Qwen" else "Phi-4-mini")
    ax.set_xlabel("Number of Gaussian-noise training variants")
    ax.set_xticks(range(1, 8))
    ax.grid(alpha=0.2)

axes[0].set_ylabel("Held-out ROC AUC gain over random selection")
axes[1].legend()

fig.suptitle(
    "Selecting Border Inputs using Gaussian noise only\n"
    "3 inputs × 10 detection observations per test"
)

# Save in both model-analysis folders.
for folder in collections.values():
    output = folder / "extension_single_family_v1"
    for extension in ["png", "pdf"]:
        fig.savefig(
            output / f"noise_training_variant_budget_gain.{extension}",
            dpi=300,
            bbox_inches="tight",
        )

plt.show()

print(
    "Points: average across all training-variant combinations, "
    "then across five seeds.\n"
    "Error bars: ±1 SD of paired gains across seeds, "
    "not confidence intervals.\n"
    "Variation between variant combinations is not shown."
)

## Descriptive entropy–AUC Spearman correlations

In [ ]:
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

correlation_rows = []
input_rows = []

for model, folder in collections.items():
    output = folder / "extension_single_family_v1"
    protocol = json.loads((output / "protocol.json").read_text())
    prompts = protocol["candidate_prompts"]

    for seed in protocol["seeds"]:
        saved = json.loads(
            (output / f"selection_scores_seed_{seed}.json").read_text()
        )
        assert saved["protocol"] == protocol

        entropy = np.array([
            saved["entropy_bits"][prompt] for prompt in prompts
        ])

        for family in protocol["families"]:
            scores = np.array([
                saved["family_ranking_scores"][family][prompt]
                for prompt in prompts
            ])

            # Correlation is undefined for a constant vector.
            if np.unique(entropy).size < 2 or np.unique(scores).size < 2:
                rho = np.nan
            else:
                rho = float(spearmanr(entropy, scores).statistic)

            correlation_rows.append({
                "model": model,
                "seed": seed,
                "training_family": family,
                "spearman_rho": rho,
            })

            for prompt, h, auc in zip(prompts, entropy, scores):
                input_rows.append({
                    "model": model,
                    "seed": seed,
                    "training_family": family,
                    "prompt": prompt,
                    "entropy_bits": h,
                    "selection_auc": auc,
                })

correlations = pd.DataFrame(correlation_rows)
input_scores = pd.DataFrame(input_rows)

correlation_summary = (
    correlations.groupby(
        ["model", "training_family"], sort=False
    )
    .agg(
        mean_rho=("spearman_rho", "mean"),
        min_rho=("spearman_rho", "min"),
        max_rho=("spearman_rho", "max"),
        valid_seeds=("spearman_rho", "count"),
    )
    .reset_index()
)

for model, folder in collections.items():
    output = folder / "extension_single_family_v1"

    for frame, filename in [
        (correlations, "entropy_spearman_per_seed.csv"),
        (correlation_summary, "entropy_spearman_summary.csv"),
        (input_scores, "entropy_and_selection_scores.csv"),
    ]:
        frame[frame["model"] == model].to_csv(
            output / filename, index=False
        )

display(correlation_summary.round(4))
print(
    "Descriptive correlations across ten inputs per seed.\n"
    "Seeds reuse the same recorded collection; "
    "the range is not a confidence interval.\n"
    "These correlations do not establish a causal explanation."
)

## Full transfer grid with observed combination/seed ranges

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

family_labels = {
    "magnitude": "Magnitude pruning",
    "random": "Random pruning",
    "random_noise": "Gaussian noise",
}
families = list(family_labels)

for model, folder in collections.items():
    output = folder / "extension_single_family_v1"
    data = pd.read_csv(
        output / "variant_budget_curve_per_seed.csv"
    )

    fig, axes = plt.subplots(
        2, 3, figsize=(13, 7),
        sharey=True, constrained_layout=True,
    )

    for column, train in enumerate(families):
        tests = [family for family in families if family != train]

        for row, test in enumerate(tests):
            ax = axes[row, column]
            part = data[
                (data["training_family"] == train)
                & (data["test_family"] == test)
            ].copy()

            # Subtract the random baseline from the same seed/test.
            part["min_gain"] = (
                part["min_combination_auc"] - part["random_mean_auc"]
            )
            part["max_gain"] = (
                part["max_combination_auc"] - part["random_mean_auc"]
            )

            stats = (
                part.groupby("n_training_variants")
                .agg(
                    mean_gain=("gain_vs_random", "mean"),
                    seed_sd=("gain_vs_random", "std"),
                    envelope_min=("min_gain", "min"),
                    envelope_max=("max_gain", "max"),
                )
            )

            x = stats.index.to_numpy()

            ax.fill_between(
                x,
                stats["envelope_min"].to_numpy(),
                stats["envelope_max"].to_numpy(),
                color="#2878B5", alpha=0.15,
                label="Observed combination/seed range",
            )
            ax.errorbar(
                x,
                stats["mean_gain"],
                yerr=stats["seed_sd"],
                color="#2878B5", marker="o", capsize=3,
                label="Mean ± seed SD",
            )

            ax.axhline(0, color="gray", linestyle="--", linewidth=1)
            ax.set_title(
                f"Train: {family_labels[train]}\n"
                f"Test: {family_labels[test]}",
                fontsize=10,
            )
            ax.set_xticks(x)
            ax.set_xlabel("Number of training variants")
            ax.grid(alpha=0.2)

    for ax in axes[:, 0]:
        ax.set_ylabel("ROC AUC gain over random selection")

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="outside lower center",
        ncol=2, frameon=False,
    )
    fig.suptitle(
        f"{model}: transfer across perturbation families\n"
        "Fixed detection budget: 3 inputs × 10 observations"
    )

    for extension in ["png", "pdf"]:
        fig.savefig(
            output / f"all_family_budget_curves.{extension}",
            dpi=300, bbox_inches="tight",
        )

    plt.show()

print(
    "Band: observed min–max across variant combinations and seeds.\n"
    "Bars: ±1 SD across seeds of combination-averaged gains.\n"
    "Neither represents a confidence interval."
)